# Using SUPREME with your own components

This notebook takes the perspective of an **external user**: you `pip install`
SUPREME into your own project and plug in your **own datasets, models, unlearning
methods, baselines and metrics** - without editing any framework code.

SUPREME is registry-based. A component is resolved by *name* through a convention
(`Foo` -> `supreme.<subpackage>.Foo`), and you extend it by **registering a module
path** that points at *your* code. Resolution order is **runtime overrides ->
packaging entry points -> built-in convention**, so your components never collide
with or alter the built-ins. The full component interfaces and the Lightning Fabric
integration rules live in [`docs/extending.md`](../docs/extending.md).

## 1. Install SUPREME into your project

In your own project's environment:

```bash
pip install supreme            # core (CPU / MPS)
pip install "supreme[cuda]"    # + DeepSpeed / bitsandbytes / NVIDIA telemetry
```

`import supreme` is lightweight - it does **not** import the PyTorch/Lightning stack,
so registration works even before the heavy dependencies are touched.

In [ ]:
import supreme
print('SUPREME', supreme.__version__)
print('register API:', [n for n in supreme.__all__ if n.startswith('register')])

## 2. A guaranteed-runnable proof (no GPU needed)

Registration and resolution are torch-free, so this cell runs anywhere. We write a
tiny external module, register a method and a model from it, and confirm SUPREME
resolves the names to *our* module path and can load them. (Real components are
torch-based - that's section 3.)

In [ ]:
import os, sys, importlib, tempfile, textwrap

# Create a throwaway external package on sys.path (stands in for your pip-installed pkg)
demo_dir = tempfile.mkdtemp()
sys.path.insert(0, demo_dir)
with open(os.path.join(demo_dir, 'acme_extras.py'), 'w') as f:
    f.write(textwrap.dedent('''
        def acme_method(fabric=None, model=None, **kwargs):
            # your unlearning logic; return value is ignored by the framework
            return "acme_method ran"

        def AcmeNet(num_labels=10, **kwargs):
            return {"kind": "AcmeNet", "num_labels": num_labels}
    '''))

supreme.register_unlearning_method('acme_method', 'acme_extras')        # attr defaults to name
supreme.register_model('AcmeNet', 'acme_extras:AcmeNet')                # explicit module:attr

from supreme import registry as R
print('method ->', R.resolve_method_location('acme_method'))
print('model  ->', R.resolve_callable_location('model', 'AcmeNet'))

def _load(loc):
    mod, attr = loc
    return getattr(importlib.import_module(mod), attr)
print('call method:', _load(R.resolve_method_location('acme_method'))())
print('call model :', _load(R.resolve_callable_location('model', 'AcmeNet'))(num_labels=7))

# Registration also syncs the framework's name lists, so the CLI/validation accept them:
print('in all_methods :', 'acme_method' in supreme.project_config.all_methods)
print('in model_names :', 'AcmeNet' in supreme.project_config.model_names)

## 3. Write your real components

These follow the documented interfaces (see [`docs/extending.md`](../docs/extending.md)).
In your project they live in normal modules of your package; here we use
`%%writefile` so the notebook is self-contained. They are torch-based, so running
them requires SUPREME's full stack installed.

### 3a. A custom unlearning method

Signature: the framework always passes `fabric`, `model`, `model_name`,
`num_gpus`, `wandb_logging_flag`, `type_of_unlearning_strategy`, plus the standard
retain/forget train/test dataloaders. Pair every model with its optimiser through
`fabric.setup(...)`, and use `fabric.backward(loss)` for gradient sync.

In [ ]:
%%writefile my_extras_method.py
import torch
import torch.nn as nn

def gentle_finetune(fabric, model, retain_train_dataloader, lr=0.01, epochs=1, **kwargs):
    """Toy method: fine-tune on the retain set so the model forgets the rest."""
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    model, optimizer = fabric.setup(model, optimizer)
    model.train()
    for _ in range(epochs):
        for images, labels in retain_train_dataloader:
            optimizer.zero_grad()
            loss = nn.functional.cross_entropy(model(images), labels)
            fabric.backward(loss)   # NOT loss.backward() - Fabric syncs grads
            optimizer.step()
    # nothing needs to be returned

### 3b. A custom evaluation metric

Decorate with `@track_evaluation_metric` so the metric returns the standard result
envelope and gets memory/power/time tracking. Aggregate across processes with
`fabric.all_gather(...)`. Metric names not recognised by the built-in dispatcher are
resolved through the registry and run automatically - **no edits to
`metrics_main.py`**. Pass `requires_retrain=True` at registration time if the metric
needs the retrained reference model.

In [ ]:
%%writefile my_extras_metric.py
import torch
from supreme.utils.unlearning.evaluation_utils import track_evaluation_metric

@track_evaluation_metric
def forget_accuracy(fabric, unlearned_model, forget_test_dataloader, **kwargs):
    """Accuracy on the forget set (lower is better after unlearning)."""
    unlearned_model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in forget_test_dataloader:
            preds = unlearned_model(images).argmax(dim=1)
            correct += (preds == labels).sum()
            total += labels.numel()
    acc = correct.float() / max(int(total), 1)
    return fabric.all_gather(acc).mean().item()

### 3c. A custom model and dataset

A model is a factory function returning an `nn.Module`. A dataset is a class taking
the framework's constructor kwargs (`root`, `download`, `train`, `unlearning`,
`img_size`, `model_name`); subclassing an existing builder in
`supreme/datasets/datasets.py` is the easiest route. Register the dataset with its
data `root`, the `class_dict` (class-name -> integer label) used by class/subclass
unlearning, and optionally the per-architecture training schedule.

In [ ]:
%%writefile my_extras_model.py
import torch.nn as nn

def TinyCNN(num_labels, **kwargs):
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        nn.Flatten(), nn.Linear(16, num_labels),
    )

## 4. Register everything (runtime API)

Call these once, before launching the pipeline (e.g. at the top of your run script).

In [ ]:
import supreme

supreme.register_unlearning_method('gentle_finetune', 'my_extras_method')
supreme.register_metric('forget_accuracy', 'my_extras_metric', requires_retrain=False)
supreme.register_model('TinyCNN', 'my_extras_model:TinyCNN')
# A dataset example (points at your Dataset subclass):
supreme.register_dataset(
    'MyFaces', 'my_extras_dataset:MyFaces',
    root='/data/myfaces',
    class_dict={'alice': 0, 'bob': 1, 'carol': 2},
    rn_epochs=100, rn_milestones=[30, 60, 80],   # ResNet schedule
    vit_epochs=8,  vit_milestones=[7],            # ViT schedule
)

pc = supreme.project_config
print('method  registered:', 'gentle_finetune' in pc.all_methods)
print('metric  registered:', 'forget_accuracy' in pc.evaluation_metrics)
print('model   registered:', 'TinyCNN' in pc.model_names)
print('dataset registered:', 'MyFaces' in pc.dataset_names)

## 5. Alternative: ship a plugin via packaging entry points

Instead of calling `register_*` at runtime, an installed package can advertise its
components declaratively; SUPREME discovers them automatically on first use. Use the
direct `module:attribute` groups for the callable categories:

```toml
# in your plugin package's pyproject.toml
[project.entry-points."supreme.unlearning_methods"]
gentle_finetune = "my_extras_method:gentle_finetune"

[project.entry-points."supreme.metrics"]
forget_accuracy = "my_extras_metric:forget_accuracy"

[project.entry-points."supreme.models"]
TinyCNN = "my_extras_model:TinyCNN"
```

Datasets carry extra metadata (root, class dict, schedule), and you may want to
register several components at once, so point the `supreme.plugins` group at a
zero-argument **setup callable** that registers them via the runtime API:

```toml
[project.entry-points."supreme.plugins"]
my_extras = "my_extras_register:setup"
```

In [ ]:
%%writefile my_extras_register.py
import supreme

def setup():
    """Discovered via the 'supreme.plugins' entry-point group on first use."""
    supreme.register_dataset(
        'MyFaces', 'my_extras_dataset:MyFaces',
        root='/data/myfaces',
        class_dict={'alice': 0, 'bob': 1, 'carol': 2},
    )
    supreme.register_unlearning_method('gentle_finetune', 'my_extras_method')

## 6. Run the pipeline with your components

Your registered names are now first-class: pass them to the CLI, the console
scripts, or the Python API exactly like built-ins. (Requires the full stack, a GPU,
and a trained checkpoint - see [`docs/script_arguments.md`](../docs/script_arguments.md).)

In [ ]:
# import supreme
# supreme.run_unlearning([
#     '-method', 'gentle_finetune',
#     '-net', 'TinyCNN',
#     '-dataset', 'MyFaces',
#     '-eval_metrics', 'accuracy,forget_accuracy',   # built-in + your metric
#     '-type_of_unlearning_strategy', 'fullclass',
#     '-seed', '260', '-precision', '32-true',
#     '-weight_path', '<path/to/trained.ckpt>',
# ])
#
# Equivalent on the command line:
#   supreme-unlearn -method gentle_finetune -net TinyCNN -dataset MyFaces \
#       -eval_metrics accuracy,forget_accuracy -type_of_unlearning_strategy fullclass \
#       -seed 260 -precision 32-true -weight_path <path/to/trained.ckpt>